# GUI Element Detector - train on Google Colab

Use this if your computer has no NVIDIA GPU.

1. In Colab: **Runtime > Change runtime type > T4 GPU**.
2. Zip your `gui-detector` folder **without** `.venv`, `data/` and `results/` (only `src/`, `app.py`, `requirements.txt` are needed) and name it `gui-detector.zip`.
3. Run the cells below in order. The first cell asks you to upload the zip.

Training takes roughly 1-1.5 hours per model on a T4.

In [ ]:
from google.colab import files
up = files.upload()  # choose gui-detector.zip
!unzip -o -q gui-detector.zip -d /content/
%cd /content/gui-detector
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install ultralytics pandas opencv-python

## 1. Download VINS and convert it to YOLO format

In [ ]:
!mkdir -p data/raw
!gdown 1ucti2ujNJgKV-Ts_mXALA7BZ97iPv8h6 -O data/raw/vins.zip
!cd data/raw && unzip -q vins.zip && rm -rf vins.zip __MACOSX
!python src/prepare_data.py

## 2. Train (yolov8s main model, yolov8n comparison)

In [ ]:
!python src/train.py --model yolov8s.pt
!python src/train.py --model yolov8n.pt

## 3. Evaluate on the test split

In [ ]:
!cp results/yolov8s_640_best.pt results/best.pt
!python src/evaluate.py --weights results/yolov8n_640_best.pt
!python src/evaluate.py --weights results/best.pt --name yolov8s_640 --main
import pandas as pd
display(pd.read_csv('results/comparison.csv'))
display(pd.read_csv('results/metrics.csv'))

## 4. Optional: higher resolution for small elements

In [ ]:
!python src/train.py --model yolov8s.pt --imgsz 960
!python src/evaluate.py --weights results/yolov8s_960_best.pt --imgsz 960

## 5. Download results (copy `results/` into your local `gui-detector/` folder)

In [ ]:
!zip -q -r results.zip results -x 'results/runs/*/weights/last.pt'
files.download('results.zip')